In [1]:
#!/usr/bin/env python3
"""
Compare performance across scenarios for a given notebook ID.

Usage:
  python compare_metrics.py --input /path/to/metrics.xlsx --notebook-id 2 --outdir ./outputs

Outputs:
  - notebook_<ID>_filtered.csv
  - notebook_<ID>_summary.csv
  - notebook_<ID>_solve_time.png
  - notebook_<ID>_build_time.png
  - notebook_<ID>_initialization_time.png
  - notebook_<ID>_total_wall_time.png
"""

import argparse
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ----------------------------
# Column normalization helpers
# ----------------------------
def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Standardize column names to:
        notebook_id, scenario, solve_time, build_time, initialization_time, total_wall_time
    Accepts common variants (case-insensitive). Missing columns are created as NaN.
    """
    cols_lower = {c.lower(): c for c in df.columns}
    colmap = {}

    def pick(name_variants):
        for cand in name_variants:
            key = cand.lower()
            if key in cols_lower:
                return cols_lower[key]
        return None

    m = pick(["notebook_id", "notebook id", "nb_id", "notebook", "id"])
    if m: colmap[m] = "notebook_id"

    m = pick(["scenario", "method", "case", "strategy"])
    if m: colmap[m] = "scenario"

    m = pick(["solve_time", "solve time", "solve", "solve_time_s", "solve(s)", "solve_sec", "solve seconds"])
    if m: colmap[m] = "solve_time"

    m = pick(["build_time", "build time", "build", "build_time_s", "build(s)", "build_sec", "build seconds"])
    if m: colmap[m] = "build_time"

    m = pick(["initialization time", "init_time", "init time", "initialization_time", "init", "initialization", "Init_time"])
    if m: colmap[m] = "initialization_time"

    m = pick(["total wall time", "total_wall_time", "tot_wlclk", "wall-clock time", "wall time", "wall_clock", "total_time"])
    if m: colmap[m] = "total_wall_time"

    df2 = df.rename(columns=colmap).copy()

    # Ensure required columns exist
    needed = ["notebook_id", "scenario", "solve_time", "build_time", "initialization_time", "total_wall_time"]
    for n in needed:
        if n not in df2.columns:
            df2[n] = np.nan

    return df2


def canonicalize_scenario(s):
    """Map common aliases to 'central' or 'symbolic' for clean comparisons."""
    if not isinstance(s, str):
        return s
    s_low = s.strip().lower()

    central_aliases = {
        "central", "central difference", "central_difference", "fd_central",
        "finite difference (central)", "finite difference", "fd", "central-diff",
        "central diff", "central fd"
    }
    symbolic_aliases = {
        "symbolic", "symbolic differentiation", "sym", "symbolic_diff", "symbolic-diff"
    }

    if s_low in central_aliases:
        return "central"
    if s_low in symbolic_aliases:
        return "symbolic"
    return s.strip()


# ----------------------------
# Plotting
# ----------------------------
def plot_metric(df_mean: pd.DataFrame, metric_col: str, title: str, outpath: Path):
    """
    Single-metric bar chart across scenarios (prefers central, symbolic order).
    """
    preferred_order = ["central", "symbolic"]
    available = df_mean["scenario"].tolist()
    order = [s for s in preferred_order if s in available] + [s for s in available if s not in preferred_order]
    sub = df_mean.set_index("scenario").loc[order].reset_index()

    plt.figure(figsize=(6, 4))
    plt.bar(sub["scenario"], sub[metric_col])
    plt.title(title)
    plt.ylabel("seconds")
    plt.xlabel("Scenario")
    # Keep labels horizontal for readability (user previously requested readable x-axis labels)
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.savefig(outpath, dpi=200)
    plt.close()


# ----------------------------
# Main
# ----------------------------
def main():
    parser = argparse.ArgumentParser(description="Compare scenario performance for a given notebook ID.")
    parser.add_argument("/Users/snarasi2/projects/Pyomo_DoE/Pyomo.DoE/outputs/metrics.xlsx", type=str, required=True, help="Path to metrics.xlsx")
    parser.add_argument("2", type=int, required=True, help="Notebook ID to filter (e.g., 2)")
    parser.add_argument("--outdir", type=str, default=".", help="Output directory (default: current dir)")
    args = parser.parse_args()

    in_path = Path(args.input)
    out_dir = Path(args.outdir)
    out_dir.mkdir(parents=True, exist_ok=True)

    if not in_path.exists():
        raise FileNotFoundError(f"Input file not found: {in_path}")

    # Load and normalize
    raw = pd.read_excel(in_path)
    df = normalize_columns(raw)
    df["scenario"] = df["scenario"].apply(canonicalize_scenario)

    # Filter to notebook ID; if empty and 'notebook_id' is missing/NaN everywhere, treat entire sheet as one set
    df_nb = df[df["notebook_id"] == args.notebook_id].copy()
    if df_nb.empty and df["notebook_id"].notna().any():
        # Selected ID not present
        print(f"[WARN] No rows found for notebook_id={args.notebook_id}.")
    elif df_nb.empty:
        # No notebook_id info at all; fallback to all data
        print("[INFO] No valid 'notebook_id' column; using all rows.")
        df_nb = df.copy()

    # Save filtered
    filtered_csv = out_dir / f"notebook_{args.notebook_id}_filtered.csv"
    df_nb.to_csv(filtered_csv, index=False)

    # Aggregate (mean/std) per scenario
    metrics = ["solve_time", "build_time", "initialization_time", "total_wall_time"]
    agg_mean = (
        df_nb.groupby("scenario", dropna=False)[metrics]
        .mean(numeric_only=True)
        .reset_index()
    )
    agg_std = (
        df_nb.groupby("scenario", dropna=False)[metrics]
        .std(numeric_only=True)
        .reset_index()
    )
    summary = agg_mean.merge(agg_std, on="scenario", suffixes=("_mean", "_std"))

    # Save summary
    summary_csv = out_dir / f"notebook_{args.notebook_id}_summary.csv"
    summary.to_csv(summary_csv, index=False)

    # Plots
    plot_metric(
        agg_mean,
        "solve_time",
        f"Solve Time — Notebook {args.notebook_id}",
        out_dir / f"notebook_{args.notebook_id}_solve_time.png",
    )
    plot_metric(
        agg_mean,
        "build_time",
        f"Build Time — Notebook {args.notebook_id}",
        out_dir / f"notebook_{args.notebook_id}_build_time.png",
    )
    plot_metric(
        agg_mean,
        "initialization_time",
        f"Initialization Time — Notebook {args.notebook_id}",
        out_dir / f"notebook_{args.notebook_id}_initialization_time.png",
    )
    plot_metric(
        agg_mean,
        "total_wall_time",
        f"Total Wall Time — Notebook {args.notebook_id}",
        out_dir / f"notebook_{args.notebook_id}_total_wall_time.png",
    )

    print(f"[OK] Wrote: {filtered_csv}")
    print(f"[OK] Wrote: {summary_csv}")
    print(f"[OK] Charts saved to: {out_dir.resolve()}")


if __name__ == "__main__":
    main()


TypeError: 'required' is an invalid argument for positionals